# Gold Layer — fact_sales

Builds the `workspace.gold.fact_sales` fact table from `workspace.silver.sales_details`, enriching it with surrogate keys from the gold dimension tables.

---

## Source & Join Logic

| Source Table | Layer | Role | Join Key |
|---|---|---|---|
| `workspace.silver.sales_details` | Silver | Base fact grain (one row per order line) | — |
| `workspace.gold.dim_customer` | Gold | Adds `customer_id` (surrogate PK) | `sales_details.customer_id = dim_customer.customer_id` |
| `workspace.gold.dim_product` | Gold | Adds `product_id` (surrogate PK, replacing business key `product_key`) | `sales_details.product_key = dim_product.product_key` (direct match — prefix already stripped in dim_product) |

---

## Keys Added from Dimension Tables

| Key | Source | Type | Notes |
|---|---|---|---|
| `customer_id` | dim_customer | int | Already present in silver; validated against dim_customer to ensure referential integrity |
| `product_id` | dim_product | int | **New** — surrogate key pulled from dim_product via join on `product_key`; replaces the string business key |

### Product Key Matching

`dim_product.product_key` is now normalised to the short form (e.g., `HL-U509-R`), matching `sales_details.product_key` directly. `dim_product` is deduplicated to one row per `product_key` (397→295 rows), so a simple LEFT JOIN gives a clean 1:1 match with no row explosion.

The original `product_key` (string business key from silver) is retained for traceability, but `product_id` is the proper FK to `dim_product`.

---

## Measures

`sales_amount`, `quantity`, `price` are carried directly from silver as fact measures.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.fact_sales
COMMENT 'Gold-layer sales fact table from silver.sales_details, enriched with surrogate keys from dim_customer and dim_product.'
AS
SELECT
    -- Surrogate key for the fact table itself
    ROW_NUMBER() OVER (ORDER BY sd.order_number, sd.product_key) AS fact_sales_key,

    -- Foreign keys to dimension tables
    dc.customer_id       AS customer_id,    -- FK → dim_customer.customer_id (surrogate PK)
    dp.product_id        AS product_id,     -- FK → dim_product.product_id (surrogate PK, new)
    sd.product_key       AS product_key,    -- original business key retained for traceability

    -- Business identifiers
    sd.order_number,

    -- Date attributes
    sd.order_date,
    sd.ship_date,
    sd.due_date,

    -- Measures
    sd.sales_amount,
    sd.quantity,
    sd.price

FROM workspace.silver.sales_details sd
LEFT JOIN workspace.gold.dim_customer dc
    ON sd.customer_id = dc.customer_id
LEFT JOIN workspace.gold.dim_product dp
    ON sd.product_key = dp.product_key

In [0]:
%sql
SELECT * FROM workspace.gold.fact_sales LIMIT 10